# 12 — Entity extraction evaluation gates

This notebook separates **operational integrity**, **synthetic implementation demonstrations**, and **held-out accuracy**. The current unlabelled A2 documents cannot supply precision, recall, F1, normalisation accuracy or ranking gains.

It independently rebuilds the same 1,000-training-CV / 1,000-training-JD run (seed 0). Package tests cover punctuation, ambiguous Go/R, overlap handling, negation/learning, unchanged source text and train/evaluation separation. Passing these invariants does not measure corpus accuracy.

In [1]:
import json
import pandas as pd
from candidate_screener.config import PROJECT_ROOT

# Aggregate outputs only: cap every saved output at 2,000 characters.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")

SAMPLE_SIZE = 1000  # per document type, exclusively from the frozen A2 train region
SEED = 0


In [2]:
from candidate_screener.extraction.run import run_extraction, validate_run

result = run_extraction(sample_size=SAMPLE_SIZE, seed=SEED)
integrity = validate_run(result, sample_size=SAMPLE_SIZE, seed=SEED)
show_summary(integrity)


{
  "entities_checked": 6598,
  "unique_entity_ids": true,
  "exact_field_offsets": true,
  "training_only": true,
  "reproduces": true,
  "entities_sha256": "8ad1de2255d310ca9b014e287895f8deb8424ce65d18d9f420f01d4e64ac22cd"
}


## Named operational probes on synthetic text

These independently authored probes check punctuation preservation, ordinary Go avoidance and negated/learning contexts. They test implementation behaviour rather than corpus accuracy. Original text is used for offsets; only bounded boolean summaries are displayed.

In [3]:
from candidate_screener.extraction import extract_document
from candidate_screener.annotation.redact import redact_text, assert_clean

probe_text = "No Python experience. Learning Kubernetes. C++, C#, .NET and Node.js. We go home."
probe_document = {"doc_id": "synthetic-probes", "source": "synthetic", "doc_type": "cv", "fields": {"Moreinfo": probe_text}}
probe_entities = extract_document(probe_document)
probe_surfaces = {e["surface"] for e in probe_entities}
probe_statuses = {e["surface"]: e["context_status"] for e in probe_entities}
checks = {
    "punctuation_sensitive_mentions": {"C++", "C#", ".NET", "Node.js"}.issubset(probe_surfaces),
    "ordinary_go_not_extracted": not any(e["surface"].lower() == "go" for e in probe_entities),
    "python_negated": probe_statuses.get("Python") == "negated",
    "kubernetes_learning": probe_statuses.get("Kubernetes") == "learning",
    "unchanged_field_offsets": all(probe_text[e["start"]:e["end"]] == e["surface"] for e in probe_entities),
}
assert all(checks.values()), checks
assert_clean(pd.Series([redact_text(probe_text)]), "synthetic invariant probes")
show_summary(checks)


{
  "punctuation_sensitive_mentions": true,
  "ordinary_go_not_extracted": true,
  "python_negated": true,
  "kubernetes_learning": true,
  "unchanged_field_offsets": true
}


## Exact span-and-label scorer: synthetic demonstration only

The scorer compares original field-relative spans and labels. An exact match requires document, field, start, end and label agreement; canonical IDs are not evaluated here. This tiny authored example shows scorer mechanics only. Its scores are **not accuracy estimates** for Djinni, DataTurks, SkillSpan or screening.

In [4]:
from candidate_screener.annotation.redact import redact_text, assert_clean
from candidate_screener.extraction import extract_document, score_exact_spans

synthetic_text = "Python and Kubernetes"
synthetic_document = {"doc_id": "synthetic-score", "source": "synthetic", "doc_type": "cv", "fields": {"Moreinfo": synthetic_text}}
predicted = extract_document(synthetic_document)
# Gold is authored independently for these exact synthetic spans.
gold = [
    {"doc_id": "synthetic-score", "source": "synthetic", "field": "Moreinfo", "start": 0, "end": 6, "label": "KNOWLEDGE"},
    {"doc_id": "synthetic-score", "source": "synthetic", "field": "Moreinfo", "start": 11, "end": 21, "label": "KNOWLEDGE"},
]
assert all(synthetic_text[e["start"]:e["end"]] == e["surface"] for e in predicted)
safe_example = pd.Series([redact_text(synthetic_text)])
assert_clean(safe_example, "synthetic scorer example")
show_summary({"scope": "synthetic scorer demonstration only", "metrics": score_exact_spans(gold, predicted)})


{
  "scope": "synthetic scorer demonstration only",
  "metrics": {
    "micro": {
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "support": 2,
      "predicted": 2,
      "true_positive": 2
    },
    "per_label": {
      "KNOWLEDGE": {
        "precision": 1.0,
        "recall": 1.0,
        "f1": 1.0,
        "support": 2,
        "predicted": 2,
        "true_positive": 2
      }
    }
  }
}


## Supervised evaluation prerequisites

Availability alone does not make a source ready to score. The labelled-source adapter interface remains pending for absent B1/B2 data. No fabricated zero F1 is returned. Even if restored locally, the sources require their respective schema/provenance checks and frozen rules before held-out evaluation.

In [5]:
show_summary({key: value for key, value in result["availability"].items() if key in {"djinni-cv", "djinni-jd"}})

{
  "djinni-cv": {
    "catalog_id": "A2",
    "missing_artifacts": [],
    "status": "present_unvalidated"
  },
  "djinni-jd": {
    "catalog_id": "A2",
    "missing_artifacts": [],
    "status": "present_unvalidated"
  }
}


In [6]:
show_summary({key: value for key, value in result["availability"].items() if key not in {"djinni-cv", "djinni-jd"}})

{
  "data-jobs": {
    "catalog_id": "D2",
    "missing_artifacts": [
      "*-train-*.parquet"
    ],
    "status": "missing"
  },
  "dataturks": {
    "catalog_id": "B1",
    "missing_artifacts": [
      "traindata.json",
      "testdata.json"
    ],
    "status": "missing"
  },
  "esco": {
    "catalog_id": "D1",
    "missing_artifacts": [
      "skills_en.csv",
      "occupations_en.csv",
      "occupationSkillRelations_en.csv"
    ],
    "status": "missing"
  },
  "skillspan": {
    "catalog_id": "B2",
    "missing_artifacts": [
      "*-train-*.parquet",
      "*-validation-*.parquet",
      "*-test-*.parquet"
    ],
    "status": "missing"
  }
}


In [7]:
show_summary({
    "gold_evaluation": "blocked: validated B1/B2 adapters and frozen held-out evaluation are pending",
    "canonical_mapping_accuracy": "blocked: independently labelled canonical mappings are absent",
    "ranking_benefit": "not measured: requires a separate retrieval comparison",
})

{
  "gold_evaluation": "blocked: validated B1/B2 adapters and frozen held-out evaluation are pending",
  "canonical_mapping_accuracy": "blocked: independently labelled canonical mappings are absent",
  "ranking_benefit": "not measured: requires a separate retrieval comparison"
}


## Required future adapter contracts

- **B1 DataTurks:** convert inclusive offsets; verify surfaces; repair only bounded unambiguous errors and quarantine the rest; preserve original annotated skill blobs for scoring; handle PII; freeze document-group folds. Individual predicted skills cannot be scored against automatically segmented blobs as individual gold. Report the granularity mismatch, per-class support and document-group CV mean/std.
- **B2 SkillSpan:** check BIO legality, retain independent layers, preserve published splits and audit duplicate leakage. Report each source/layer separately with support.
- Both adapters must emit `doc_id`, `field`, `start`, `end` and `label` with immutable text and provenance. Source/layer keys must agree with predictions. Never pool sources or instruments.

The run's aggregate coverage is useful for inspecting scope and computational readiness. To assess candidate-screening usefulness, first establish held-out extraction quality, then measure downstream Recall@10 separately for `generic` and `targeted`. Any future Precision@k travels with its ceiling and the documented A13 uncertainty. No new manual annotation is authorized here.